# Calibrating the Crowd — data tour

One-stop entry point to the datasets. Run top to bottom; every table is documented in `data/DATA_DICTIONARY.md`.

Core question: are prediction-market prices (Kalshi, Polymarket) as good as the de-vigged sportsbook consensus on the same games?

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

core = pd.read_csv('../data/processed/analysis_core.csv')
print(core.shape)
core.head(3)

## The headline: three venues, one Brier

`clean_set` filters to games passing all outcome cross-checks. The paper's three-way sample is the 5,328 games where all three venues priced.

In [ ]:
t3 = core[core.clean_set & core.kalshi_home_prob.notna()
          & core.polymarket_home_prob.notna() & core.book_home_prob_devig.notna()]
for c in ['kalshi_home_prob','polymarket_home_prob','book_home_prob_devig','pinnacle_home_prob']:
    sub = t3[t3[c].notna()]
    print(f"{c:>26}: n={len(sub):,}  Brier={((sub[c]-sub.home_won)**2).mean():.4f}")

## Table 1: do teams priced at X% win X% of the time?

In [ ]:
bins = np.linspace(0, 1, 11)
for c, lab in [('kalshi_home_prob','Kalshi'), ('polymarket_home_prob','Polymarket'),
               ('book_home_prob_devig','Books')]:
    b = pd.cut(t3[c], bins)
    tab = t3.groupby(b, observed=True).agg(n=(c,'size'), priced=(c,'mean'), won=('home_won','mean'))
    print(f"\n=== {lab} ==="); print((tab*[1,100,100]).round(1).to_string())

## Calibration picture (both sides stacked)

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
ax.plot([0,1],[0,1],'k--',alpha=.5)
for c, lab in [('kalshi_home_prob','Kalshi'),('polymarket_home_prob','Polymarket'),('book_home_prob_devig','Books')]:
    p = np.r_[t3[c], 1-t3[c]]; w = np.r_[t3.home_won, 1-t3.home_won]
    b = np.clip(np.digitize(p, bins)-1, 0, 9)
    xs=[p[b==k].mean() for k in range(10)]; ys=[w[b==k].mean() for k in range(10)]
    ax.plot(xs, ys, 'o-', label=lab, alpha=.7)
ax.set(xlabel='price', ylabel='win rate'); ax.legend(); plt.show()

## Other tables to explore

- `sportsbook_sharp_prices.csv` — 24 books per game incl. Pinnacle and Betfair Exchange (per-book quotes; de-vig with `p1/(p1+p2)`)
- `kalshi_trades_24h.csv` — 710K individual fills (who trades, when, what size)
- `kalshi_spread_prices.csv` + `sportsbook_alt_spreads.csv` — margin-of-victory ladders, both venues
- `kalshi_futures_prices.csv` + `sportsbook_outrights.csv` + `poly_futures_prices.csv` — the futures 2x2 (where calibration breaks)
- `kalshi_niche_prices.csv` — table tennis / cricket / esports / minor soccer (calibration without a benchmark)

Every number in the paper regenerates with `python -m src.make_results` (52 modules, audit-gated).

In [ ]:
trades = pd.read_csv('../data/processed/kalshi_trades_24h.csv')
print(f"{len(trades):,} fills, median {trades['count'].median():.0f} contracts "
      f"(~${(np.where(trades.taker_side=='yes',trades.yes_price,1-trades.yes_price)*trades['count']).mean():.0f} avg stake)")